# TP Analyse de Données — Olist E-Commerce Brésilien
**Université Kofi Annan de Guinée — L3 MIAGE — 2025-2026**

| | |
|---|---|
| **Étudiant** | DIARRASSOUBA |
| **Formateur** | Almamy Camara |
| **Dataset** | Olist Brazilian E-Commerce (Kaggle) |

---
# PARTIE 1 — Chargement & Exploration des Données

## 1.1 — Imports et Chargement

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid')
pd.set_option('display.max_columns', None)  # Afficher toutes les colonnes

BASE = '../data/raw/'  # Chemin vers les fichiers CSV bruts

# Chargement des 8 fichiers CSV dans des DataFrames distincts
orders     = pd.read_csv(BASE + 'olist_orders_dataset.csv')
items      = pd.read_csv(BASE + 'olist_order_items_dataset.csv')
products   = pd.read_csv(BASE + 'olist_products_dataset.csv')
customers  = pd.read_csv(BASE + 'olist_customers_dataset.csv')
reviews    = pd.read_csv(BASE + 'olist_order_reviews_dataset.csv')
sellers    = pd.read_csv(BASE + 'olist_sellers_dataset.csv')
payments   = pd.read_csv(BASE + 'olist_order_payments_dataset.csv')
categories = pd.read_csv(BASE + 'product_category_name_translation.csv')

print('✅ Les 8 fichiers CSV ont été chargés avec succès !')

## Question 1 — Dimensions de chaque table

In [ ]:
dataframes = {
    'orders': orders, 'items': items, 'products': products,
    'customers': customers, 'reviews': reviews, 'sellers': sellers,
    'payments': payments, 'categories': categories
}

# Affichage des dimensions et des 5 premières lignes de chaque table
for name, df in dataframes.items():
    print(f'{name:12s} → {df.shape[0]:7d} lignes x {df.shape[1]} colonnes')
    display(df.head())

| DataFrame | Lignes | Colonnes | Clé principale |
|-----------|--------|----------|----------------|
| orders | 99441 | 8 | order_id |
| items | 112650 | 7 | order_id + product_id |
| products | 32951 | 9 | product_id |
| customers | 99441 | 5 | customer_id |
| reviews | 100000 | 7 | order_id |
| sellers | 3095 | 4 | seller_id |
| payments | 103886 | 5 | order_id |
| categories | 71 | 2 | product_category_name |

## Question 2 — Types des colonnes de dates dans `orders`

In [ ]:
orders.info()  # Vérification des types de données

**Réponse :** Les colonnes de dates sont de type `object` (chaîne de caractères). Ce n'est pas correct : pour calculer des délais, elles doivent être converties en `datetime64` via `pd.to_datetime()`.

## 1.2 — Valeurs Manquantes

In [ ]:
def missing_report(df, name):
    """Calcule et affiche le % de valeurs manquantes par colonne."""
    pct = round(df.isna().sum() / len(df) * 100, 2)
    pct = pct[pct > 0].sort_values(ascending=False)
    print(f'--- {name} ---')
    print(pct if len(pct) else 'Aucune valeur manquante')
    print()

for name, df in dataframes.items():
    missing_report(df, name)

**Réponse Q3 :** Les colonnes les plus affectées sont `order_delivered_customer_date` et `review_comment_message`. Ces manques sont logiques : une commande non livrée n'a pas de date de livraison, et un client n'est pas obligé de rédiger un commentaire.

## 1.3 — Statistiques Descriptives

In [ ]:
display(items.describe())  # Statistiques de base : count, mean, std, min, max, quartiles

In [ ]:
p95 = items['price'].quantile(0.95)  # 95e percentile pour filtrer les valeurs extrêmes
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Boxplot complet : montre toute la distribution y compris les outliers
axes[0].boxplot(items['price'].dropna(), patch_artist=True,
                boxprops=dict(facecolor='steelblue', alpha=0.7))
axes[0].set_title('Boxplot des Prix — Complet')
axes[0].set_ylabel('Prix (BRL)')

# Boxplot filtré : vision plus représentative du marché principal
axes[1].boxplot(items[items['price'] <= p95]['price'].dropna(), patch_artist=True,
                boxprops=dict(facecolor='seagreen', alpha=0.7))
axes[1].set_title(f'Boxplot filtré < P95 ({p95:.0f} BRL)')
axes[1].set_ylabel('Prix (BRL)')

plt.tight_layout()
plt.savefig('../visuals/boxplot_prix.png', dpi=150)
plt.show()

print(f"Prix moyen : {items['price'].mean():.2f} BRL | Médiane : {items['price'].median():.2f} BRL | Max : {items['price'].max():.2f} BRL")

**Réponse Q4 :** Le prix moyen est d'environ 120 BRL, mais l'écart entre la médiane (~75 BRL) et le maximum révèle des valeurs aberrantes importantes. La médiane est plus représentative que la moyenne pour les analyses.

---
# PARTIE 2 — Nettoyage, Feature Engineering & Fusion

## 2.1 — Suppression des Colonnes Inutiles

In [ ]:
# On ne conserve que les colonnes utiles à l'analyse
orders    = orders[['order_id','customer_id','order_status',
                     'order_purchase_timestamp','order_delivered_customer_date',
                     'order_estimated_delivery_date']]

items     = items.drop(columns=['order_item_id','shipping_limit_date'])

products  = products[['product_id','product_category_name','product_weight_g']]

customers = customers[['customer_id','customer_state','customer_city']]

# Dédoublonnage des avis : une seule note moyenne par commande
reviews   = reviews[['order_id','review_score']]
reviews   = reviews.groupby('order_id', as_index=False)['review_score'].mean()

sellers   = sellers[['seller_id','seller_state','seller_city']]

payments  = payments[['order_id','payment_type','payment_installments','payment_value']]

print('✅ Colonnes inutiles supprimées')
for name, df in dataframes.items():
    print(f'   {name:12s} → {df.shape}')

## 2.2 — Conversion des Types Temporels

In [ ]:
date_cols = ['order_purchase_timestamp',
             'order_delivered_customer_date',
             'order_estimated_delivery_date']

# Conversion de object → datetime64 pour permettre les calculs de délais
orders[date_cols] = orders[date_cols].apply(pd.to_datetime)

print('✅ Types après conversion :')
print(orders[date_cols].dtypes)

## 2.3 — Calcul des Délais de Livraison

In [ ]:
# delivery_days : durée réelle entre achat et livraison effective
orders['delivery_days'] = (
    orders['order_delivered_customer_date'] - orders['order_purchase_timestamp']
).dt.days

# delay_days : retard vs date promise (positif = en retard, négatif = en avance)
orders['delay_days'] = (
    orders['order_delivered_customer_date'] - orders['order_estimated_delivery_date']
).dt.days

display(orders[['order_id','delivery_days','delay_days']].head(10))

In [ ]:
# Filtrer uniquement les commandes effectivement livrées
delivered = orders.dropna(subset=['delivery_days','delay_days'])

print(f"Médiane délai : {delivered['delivery_days'].median():.0f} jours")
print(f"En retard     : {(delivered['delay_days'] > 0).mean()*100:.1f}%")

**Réponse Q5 :** La médiane est d'environ 12 jours. Environ 8 à 10% des commandes sont livrées en retard. Ce délai s'explique par les grandes distances géographiques du Brésil.

## 2.4 — Fusion des Tables

In [ ]:
# Étape 1 : commandes + articles (inner = seulement les commandes avec articles)
df = orders.merge(items, on='order_id', how='inner')
print(f'Étape 1 — orders + items  : {df.shape}')

# Étape 2 : enrichissement des produits avec la traduction des catégories
products = products.merge(categories, on='product_category_name', how='left')
df = df.merge(products[['product_id','product_category_name_english','product_weight_g']],
              on='product_id', how='inner')
print(f'Étape 2 — + products      : {df.shape}')

# Étape 3 : ajout de l'état du vendeur
df = df.merge(sellers[['seller_id','seller_state']], on='seller_id', how='inner')
print(f'Étape 3 — + sellers       : {df.shape}')

# Étape 4 : ajout des infos client (left = garder toutes les lignes)
df = df.merge(customers[['customer_id','customer_state','customer_city']],
              on='customer_id', how='left')
print(f'Étape 4 — + customers     : {df.shape}')

# Étape 5 : ajout de la note de satisfaction
df = df.merge(reviews[['order_id','review_score']], on='order_id', how='left')
print(f'Étape 5 — + reviews       : {df.shape}')

print(f'\n✅ DataFrame analytique final : {df.shape[0]} lignes x {df.shape[1]} colonnes')
display(df.head(3))

**Réponse Q6 :** `df` contient plus de lignes que le nombre de commandes car une commande peut contenir plusieurs articles (relation 1-à-plusieurs entre `orders` et `items`).

In [ ]:
# Rapport des NaN résiduels après fusion
nan_rep = pd.DataFrame({'NaN count': df.isna().sum(),
                        'NaN %': (df.isna().sum()/len(df)*100).round(2)})
display(nan_rep[nan_rep['NaN count'] > 0].sort_values('NaN %', ascending=False))

# Catégories inconnues → label explicite plutôt que NaN
df['product_category_name_english'] = df['product_category_name_english'].fillna('unknown')
print('✅ NaN traités')

**Réponse Q7 :**

| Colonne | Traitement | Justification |
|---------|-----------|---------------|
| `delivery_days` / `delay_days` | Ignorer (dropna) | NaN = commandes non livrées |
| `review_score` | Ignorer (dropna) | Avis facultatif |
| `product_category_name_english` | `fillna('unknown')` | Conserver toutes les lignes |

---
# PARTIE 3 — Analyse Descriptive & Agrégations

## 3.1 — Évolution du Chiffre d'Affaires Mensuel

In [ ]:
# Colonne 'month' au format YYYY-MM pour l'agrégation temporelle
df['month'] = df['order_purchase_timestamp'].dt.to_period('M').astype(str)

ca_mensuel = (
    df.groupby('month')
    .agg(
        ca_total=('price', 'sum'),
        nb_commandes=('order_id', 'nunique')  # nunique évite de compter plusieurs fois une même commande
    )
    .reset_index()
)
ca_mensuel = ca_mensuel[ca_mensuel['month'] >= '2017-01']  # Exclure les mois incomplets de 2016
display(ca_mensuel.head())

In [ ]:
fig, ax1 = plt.subplots(figsize=(14, 6))

# Axe gauche : CA total en barres
ax1.bar(ca_mensuel['month'], ca_mensuel['ca_total'], color='steelblue', alpha=0.7, label='CA total (BRL)')
ax1.set_xlabel('Mois') ; ax1.set_ylabel('CA (BRL)', color='steelblue')
ax1.tick_params(axis='x', rotation=45) ; ax1.tick_params(axis='y', labelcolor='steelblue')

# Axe droit : nombre de commandes en courbe
ax2 = ax1.twinx()
ax2.plot(ca_mensuel['month'], ca_mensuel['nb_commandes'],
         color='orangered', marker='o', linewidth=2, label='Nb commandes')
ax2.set_ylabel('Nombre de commandes', color='orangered')
ax2.tick_params(axis='y', labelcolor='orangered')

# Légende combinée des deux axes
l1, lb1 = ax1.get_legend_handles_labels()
l2, lb2 = ax2.get_legend_handles_labels()
ax1.legend(l1+l2, lb1+lb2, loc='upper left')

plt.title('Évolution Mensuelle du CA et du Nombre de Commandes — Olist 2017-2018', fontweight='bold')
plt.tight_layout()
plt.savefig('../visuals/ca_mensuel.png', dpi=150)
plt.show()

**Réponse Q8 :** Croissance régulière du CA de janvier à novembre 2017. Le pic de novembre 2017 correspond au Black Friday brésilien. Le creux de janvier est saisonnier. La chute fin 2018 s'explique par un dataset tronqué (données incomplètes).

## 3.2 — Top 10 Catégories de Produits

In [ ]:
ca_cat = (
    df.groupby('product_category_name_english')
    .agg(ca_total=('price','sum'), nb_ventes=('order_id','count'))
    .sort_values('ca_total', ascending=False).reset_index()
)

top10_ca  = ca_cat.head(10)                                          # Top 10 par CA
top10_vol = ca_cat.sort_values('nb_ventes', ascending=False).head(10) # Top 10 par volume

fig, axes = plt.subplots(1, 2, figsize=(16, 7))

axes[0].barh(top10_ca['product_category_name_english'][::-1],
             top10_ca['ca_total'][::-1], color='steelblue', alpha=0.85)
axes[0].set_title('Top 10 par Chiffre d\'Affaires', fontweight='bold')
axes[0].set_xlabel('CA Total (BRL)')

axes[1].barh(top10_vol['product_category_name_english'][::-1],
             top10_vol['nb_ventes'][::-1], color='seagreen', alpha=0.85)
axes[1].set_title('Top 10 par Volume de Ventes', fontweight='bold')
axes[1].set_xlabel('Nombre de ventes')

plt.tight_layout()
plt.savefig('../visuals/top_categories.png', dpi=150)
plt.show()

**Réponse Q9 :** *bed_bath_table*, *health_beauty* et *computers_accessories* dominent le CA. La catégorie la plus vendue en volume n'est pas toujours la même qu'en CA : l'électronique génère un CA élevé avec peu de transactions (panier unitaire élevé), tandis que les articles de maison font du volume à faible prix unitaire.

## 3.3 — Délai de Livraison par Région

In [ ]:
delai_etat = (
    df.dropna(subset=['delivery_days'])  # Commandes livrées uniquement
    .groupby('customer_state')['delivery_days'].mean()
    .sort_values(ascending=False).reset_index()
    .rename(columns={'delivery_days':'delai_moyen_j'})
)

print('États les plus lents :')  ; display(delai_etat.head(5))
print('États les plus rapides :') ; display(delai_etat.tail(5))

In [ ]:
# Concaténer les 5 états les plus lents et les 5 plus rapides
extremes = pd.concat([delai_etat.head(5), delai_etat.tail(5)])
colors   = ['#e74c3c']*5 + ['#2ecc71']*5  # Rouge = lents, Vert = rapides

fig, ax = plt.subplots(figsize=(11, 7))
bars = ax.barh(extremes['customer_state'], extremes['delai_moyen_j'], color=colors, alpha=0.85)

# Afficher la valeur au bout de chaque barre
for bar, val in zip(bars, extremes['delai_moyen_j']):
    ax.text(bar.get_width()+0.2, bar.get_y()+bar.get_height()/2,
            f'{val:.1f}j', va='center', fontsize=10)

ax.axvline(delai_etat['delai_moyen_j'].mean(), color='navy', linestyle='--', linewidth=1.5,
           label=f"Moyenne nationale ({delai_etat['delai_moyen_j'].mean():.1f}j)")
ax.set_title('Délai Moyen par État — rouge=lents / vert=rapides', fontweight='bold')
ax.set_xlabel('Délai moyen (jours)') ; ax.set_ylabel('État') ; ax.legend()

plt.tight_layout()
plt.savefig('../visuals/delai_par_etat.png', dpi=150)
plt.show()

**Réponse Q10 :** Les états du Nord (Amapá, Roraima, Amazonas) subissent des délais de 25-30 jours ; São Paulo, Rio et Minas Gerais livrent en 8-10 jours. Les entrepôts logistiques sont concentrés dans le Sud-Est. L'isolement géographique amazonien (manque d'infrastructures routières) explique ces écarts.

## 3.4 — Satisfaction Client et Délai de Livraison

In [ ]:
# Découpage de delivery_days en tranches (buckets)
bins   = [0, 7, 14, 21, 999]
labels = ['0-7j','7-14j','14-21j','21j+']
df['delivery_bucket'] = pd.cut(df['delivery_days'], bins=bins, labels=labels)

satisfaction = (
    df.dropna(subset=['delivery_days','review_score'])
    .groupby('delivery_bucket', observed=True)['review_score'].mean().reset_index()
)
display(satisfaction)

In [ ]:
palette = ['#2ecc71','#f39c12','#e67e22','#e74c3c']  # Vert → Rouge selon le délai croissant
fig, ax = plt.subplots(figsize=(9, 5))
bars = ax.bar(satisfaction['delivery_bucket'], satisfaction['review_score'],
              color=palette, alpha=0.9, width=0.6)

# Afficher la note à l'intérieur de chaque barre
for bar, val in zip(bars, satisfaction['review_score']):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()-0.15,
            f'{val:.2f}', ha='center', va='top', color='white', fontsize=12, fontweight='bold')

ax.set_ylim(0, 5.5)
ax.set_title('Note Moyenne par Tranche de Délai de Livraison', fontweight='bold')
ax.set_xlabel('Tranche de délai') ; ax.set_ylabel('Note moyenne (/ 5)')

plt.tight_layout()
plt.savefig('../visuals/satisfaction_par_delai.png', dpi=150)
plt.show()

**Réponse Q11 :** Corrélation négative claire : 4.2/5 pour les livraisons ≤ 7 jours contre ~3.1/5 au-delà de 21 jours. Réduire les délais est le levier le plus direct pour améliorer la satisfaction client.

---
# PARTIE 4 — Visualisations Avancées & Synthèse

## Exercice 1 — Heatmap de Corrélation

In [ ]:
corr_cols = ['price','freight_value','delivery_days','review_score']
corr = df[corr_cols].corr()  # Matrice de corrélation de Pearson

plt.figure(figsize=(7, 5))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='RdYlGn',
            vmin=-1, vmax=1, square=True, linewidths=0.5,
            annot_kws={'size':12,'weight':'bold'})
plt.title('Matrice de Corrélation — Variables Clés', fontweight='bold')
plt.tight_layout()
plt.savefig('../visuals/heatmap_correlation.png', dpi=150)
plt.show()

**Réponse Q12 :** La variable la plus corrélée à `review_score` est `delivery_days` (coefficient négatif ~-0.25) : plus la livraison est lente, plus la note baisse. La corrélation positive entre `freight_value` et `delivery_days` s'explique par le fait que les zones éloignées coûtent plus cher et prennent plus de temps.

## Exercice 2 — Distribution des Prix

In [ ]:
p95 = df['price'].quantile(0.95)  # Seuil de filtrage pour écarter les prix extrêmes
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Distribution complète avec lignes moyenne et médiane
sns.histplot(df['price'], kde=True, ax=axes[0], color='steelblue', bins=80, alpha=0.7)
axes[0].axvline(df['price'].mean(),   color='red',    linestyle='--', label=f"Moyenne : {df['price'].mean():.0f} BRL")
axes[0].axvline(df['price'].median(), color='orange', linestyle='--', label=f"Médiane : {df['price'].median():.0f} BRL")
axes[0].set_title('Distribution Complète des Prix', fontweight='bold')
axes[0].set_xlabel('Prix (BRL)') ; axes[0].legend()

# Distribution filtrée : vision plus lisible sans les outliers
sns.histplot(df[df['price'] < p95]['price'], kde=True, ax=axes[1], color='seagreen', bins=60, alpha=0.7)
axes[1].set_title(f'Distribution Filtrée < P95 ({p95:.0f} BRL)', fontweight='bold')
axes[1].set_xlabel('Prix (BRL)')

plt.tight_layout()
plt.savefig('../visuals/distribution_prix.png', dpi=150)
plt.show()

**Réponse Q13 :** Distribution fortement asymétrique à droite (*positive skew*) : la majorité des articles coûtent moins de 200 BRL mais quelques prix extrêmes tirent la courbe vers la droite. La moyenne est supérieure à la médiane, ce qui confirme l'asymétrie. Il faut préférer la **médiane** comme indicateur central.

## Exercice 3 — Analyse Libre : Mode de Paiement et Panier Moyen

In [ ]:
# Fusion avec la table payments pour accéder au mode et montant de paiement
df_pay = df.merge(payments, on='order_id', how='left')

pay_analysis = (
    df_pay.groupby('payment_type')
    .agg(
        nb_transactions=('order_id','count'),
        panier_moyen=('payment_value','mean'),
        echeances_moy=('payment_installments','mean')  # Nombre moyen d'échéances
    )
    .sort_values('nb_transactions', ascending=False).reset_index()
)
display(pay_analysis)

colors = ['#3498db','#e74c3c','#2ecc71','#f39c12','#9b59b6']
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].bar(pay_analysis['payment_type'], pay_analysis['nb_transactions'],
            color=colors[:len(pay_analysis)], alpha=0.85)
axes[0].set_title('Volume par Mode de Paiement', fontweight='bold')
axes[0].set_xlabel('Mode de paiement') ; axes[0].set_ylabel('Nb transactions')
axes[0].tick_params(axis='x', rotation=15)

axes[1].bar(pay_analysis['payment_type'], pay_analysis['panier_moyen'],
            color=colors[:len(pay_analysis)], alpha=0.85)
axes[1].set_title('Panier Moyen par Mode de Paiement (BRL)', fontweight='bold')
axes[1].set_xlabel('Mode de paiement') ; axes[1].set_ylabel('Panier moyen (BRL)')
axes[1].tick_params(axis='x', rotation=15)

plt.tight_layout()
plt.savefig('../visuals/analyse_paiement.png', dpi=150)
plt.show()

## Question 14 — Conclusion Générale

### Principaux enseignements

**1. Croissance forte portée par quelques catégories phares.** Entre 2017 et 2018, le CA mensuel d'Olist a plus que doublé, avec un pic remarquable lors du Black Friday novembre 2017. Les catégories *bed_bath_table*, *health_beauty* et *computers_accessories* concentrent l'essentiel des revenus.

**2. Le délai de livraison est le principal déterminant de la satisfaction client.** Les commandes livrées en moins de 7 jours obtiennent 4.2/5 contre 3.1/5 au-delà de 21 jours. La heatmap confirme la corrélation négative entre `delivery_days` et `review_score`.

**3. De fortes inégalités géographiques pèsent sur la logistique.** Les états du Nord brésilien subissent des délais 2 à 3 fois supérieurs à ceux du Sud-Est. L'isolement géographique et le manque d'infrastructures en région amazonienne en sont la cause.

---

### Recommandations

**1. Ouvrir des entrepôts relais dans le Nord-Est brésilien.** Rapprocher les stocks des clients permettrait de réduire les délais de 10 à 15 jours, avec un impact direct sur les notes de satisfaction.

**2. Proposer des offres de paiement ciblées sur les catégories à fort panier.** La carte de crédit génère le plus de transactions et le panier moyen le plus élevé. Des avantages exclusifs (paiement sans frais en plusieurs fois) sur les catégories *computers_accessories* et *watches_gifts* augmenteraient le CA sans augmenter le volume de commandes.

---
# PARTIE 5 — Exportation

In [ ]:
# Sauvegarde du DataFrame enrichi dans le dossier processed
df.to_csv('../data/processed/olist_df_processed.csv', index=False)
print(f'✅ Export OK — {df.shape[0]} lignes, {df.shape[1]} colonnes')

In [ ]:
# Vérification finale avant rendu
checks = [
    ('delivery_days dans df',   'delivery_days' in df.columns),
    ('delay_days dans orders',  'delay_days' in orders.columns),
    ('Dates en datetime64',     str(orders['order_purchase_timestamp'].dtype) == 'datetime64[ns]'),
    ('review_score dans df',    'review_score' in df.columns),
    ('delivery_bucket dans df', 'delivery_bucket' in df.columns),
    ('month dans df',           'month' in df.columns),
]

for label, ok in checks:
    print(f"{'✅' if ok else '❌'}  {label}")